# Day 3 — Hands-On Lab 1: Public API + GraphDB Exploration
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Follows** | ILT 1 — API Ingestion Mechanics + Intro to GraphDB/Cypher Basics |
| **Time** | 11:00 AM – 1:00 PM (2 hours) |
| **Output** | A sandbox Delta table of FX rates, a small graph in Neo4j AuraDB, and a sandbox Delta table read back from that graph |

> **Side-exploration, not part of the GlobalMart build.** Exactly like ILT 1 said: GlobalMart's real pipeline only has two sources — Postgres CDC (Lakeflow Connect) and ADLS Autoloader. Nothing in this lab feeds Bronze/Silver/Gold or `fact_sales`. You're practicing two patterns (REST API ingestion, graph databases) that you will meet on *other* projects — GlobalMart-shaped data is just a familiar example.

### What you will build
**Part 1 — REST API (45 min):** Extend the frankfurter.app demo from ILT 1 — pull multiple base currencies, then pull historical rates for more than one date, and land both as sandbox Delta tables.

**Part 2 — GraphDB (75 min):** Create a free Neo4j AuraDB instance, seed it with a small GlobalMart-shaped graph (Customer → Order → Product → Supplier) using Cypher, query it from the Neo4j Browser, then connect to it from Databricks and land a query result as a sandbox Delta table.

**Instructions:** Run each cell with **Shift + Enter**. Cells marked `### YOUR TURN` are for you to complete — the pattern is always demonstrated once above them first.

---
## Part 1 — REST API: Beyond a Single Call

ILT 1 called `https://api.frankfurter.app/latest` once, for the default base currency (EUR), and saved the result to `sandbox_path/fx_rates`. Two realistic extensions:

1. **Multiple base currencies** — "What if GlobalMart sells in USD, GBP, *and* INR?" You need one row set per base currency, not just one.
2. **Historical rates** — "What was the rate last week?" frankfurter.app supports a date in the URL path: `https://api.frankfurter.app/2026-01-01` returns rates as of that date.

In [ ]:
# ─── Setup: same Unity Catalog External Location as ILT 1 ─────────────────────
# No storage key needed — this cluster already has access via the External
# Location's Managed Identity (set up Day 2 HOL 1).

EXTERNAL_LOCATION = "abfss://<your-container>@<your-storage-account>.dfs.core.windows.net"
sandbox_path = f"{EXTERNAL_LOCATION}/sandbox/api_graphdb"
print(f"Sandbox path: {sandbox_path}")

In [ ]:
# ─── Step 1: Multiple base currencies — one API call per currency ─────────────
# frankfurter.app takes a `base` query parameter. We loop over the currencies
# GlobalMart actually cares about and combine every response into one table.

import requests
from pyspark.sql import Row
from datetime import datetime

base_currencies = ["USD", "EUR", "GBP"]  # GlobalMart's three storefront currencies
all_rows = []

for base in base_currencies:
    response = requests.get("https://api.frankfurter.app/latest", params={"base": base})
    print(f"Called base={base} -> status {response.status_code}")

    if response.status_code != 200:
        # Don't silently skip a failed call — a DE pipeline should always know
        # when a source didn't return what was expected.
        raise RuntimeError(f"API call failed for base={base}: {response.status_code}")

    data = response.json()
    for currency, rate in data["rates"].items():
        all_rows.append(Row(
            base_currency   = data["base"],
            target_currency = currency,
            exchange_rate   = float(rate),
            rate_date       = data["date"],
            ingested_at     = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        ))

multi_base_df = spark.createDataFrame(all_rows)
print(f"\nTotal rows across {len(base_currencies)} base currencies: {multi_base_df.count()}")
multi_base_df.filter(multi_base_df.target_currency.isin("USD", "EUR", "GBP", "INR")).show()

In [ ]:
# ─── Step 2: Save the multi-currency table to sandbox ──────────────────────────
# overwrite is correct here — like ILT 1 explained, this is a live snapshot,
# not something we want to accumulate duplicate-but-stale rows for.

multi_base_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{sandbox_path}/fx_rates_multi_base")

saved = spark.read.format("delta").load(f"{sandbox_path}/fx_rates_multi_base")
print(f"Saved {saved.count()} rows to {sandbox_path}/fx_rates_multi_base")
saved.groupBy("base_currency").count().show()

### YOUR TURN — Historical Rates

frankfurter.app accepts a date in place of `latest`: `https://api.frankfurter.app/2026-01-01`.

Complete the cell below to:
1. Loop over the two dates given in `historical_dates`
2. Call the API for each date (base currency EUR is fine — keep it simple)
3. Build one combined DataFrame with a `rate_date` column that reflects the **requested** date (not `data["date"]`, in case the API adjusts to the nearest business day — compare the two and see!)
4. Append (not overwrite!) into `sandbox_path/fx_rates_historical` — this is the "accumulate history" case ILT 1 mentioned

In [ ]:
# ─── YOUR TURN: fill in the two TODOs below ────────────────────────────────────
historical_dates = ["2026-01-01", "2026-02-01"]
historical_rows  = []

for requested_date in historical_dates:
    # TODO 1: call the API for this date instead of "latest"
    # Hint: api_url = f"https://api.frankfurter.app/{requested_date}"
    api_url = None  # ← replace this line

    response = requests.get(api_url)
    print(f"Requested {requested_date} -> API returned date {response.json().get('date') if response.status_code == 200 else 'ERROR'}")

    if response.status_code != 200:
        raise RuntimeError(f"API call failed for {requested_date}: {response.status_code}")

    data = response.json()
    for currency, rate in data["rates"].items():
        historical_rows.append(Row(
            base_currency    = data["base"],
            target_currency  = currency,
            exchange_rate    = float(rate),
            requested_date   = requested_date,      # what we asked for
            api_returned_date = data["date"],        # what we actually got (may differ on weekends/holidays)
            ingested_at      = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        ))

historical_df = spark.createDataFrame(historical_rows)

# TODO 2: write this as APPEND, not overwrite — we're building history over time
historical_df.write \
    .format("delta") \
    .mode("PUT_THE_RIGHT_MODE_HERE") \
    .save(f"{sandbox_path}/fx_rates_historical")

print(f"Rows in historical table now: {spark.read.format('delta').load(f'{sandbox_path}/fx_rates_historical').count()}")

---
## Part 2 — GraphDB: Build and Query a Small GlobalMart Graph

### Step 1 — Create your free Neo4j AuraDB account and instance

You don't have a GraphDB account yet — here is exactly how to get one and end up with real, usable credentials.

1. Go to **`neo4j.com/cloud/aura`** and click **Start Free** — sign up with email or Google/GitHub (no credit card required for the Free tier).
2. Once logged in, you land on the **Aura Console**. Click **Create instance** (or **New Instance**).
3. Choose **AuraDB Free**.
4. Give it a name, e.g. `globalmart-graph-demo`.
5. Click **Create instance**. A dialog immediately shows you three values — **this is the only time the password is ever shown**:
   ```
   Connection URI : neo4j+s://xxxxxxxx.databases.neo4j.io
   Username       : neo4j
   Password       : <a long generated string>
   ```
   Click **Download credentials** (saves a small `.txt` file) or copy all three into a notes file right now. If you lose the password, you cannot recover it — you'd have to reset it from the instance's **...** menu → **Reset password**.
6. Wait ~1–2 minutes for the instance status to go from "Creating" to **Running**.
7. Click **Open** on the instance card — this launches the **Neo4j Browser**, a web UI where you can run Cypher directly. Keep this tab open; you'll come back to it.

> These 3 values (URI, username, password) are the **actual credentials** you'll paste into the notebook cell further down in Part 3 — replacing the `YOUR_INSTANCE_ID` / `YOUR_AURADB_PASSWORD` placeholders.</cell id="hol1-08">


### Step 2 — Prepare the 4 CSV files you'll upload

Instead of typing Cypher `CREATE` statements by hand, you'll build this graph the way you'd actually do it with real data: **upload CSV files** and let Neo4j's import tool turn them into nodes and relationships.

Create these 4 files on your own machine (Notepad / Excel / any text editor — save as plain `.csv`):

**`customers.csv`**
```
customer_id,name,city
CUST-001,Raj Patel,Mumbai
CUST-002,Priya Singh,Bangalore
CUST-003,Arjun Mehta,Delhi
```

**`suppliers.csv`**
```
supplier_id,name
SUP-001,TechDistributors Inc
SUP-002,HomeOffice Supplies
```

**`products.csv`** — note the `supplier_id` column; that's what lets the importer wire up `SUPPLIED_BY` automatically
```
product_id,name,category,supplier_id
PRD-001,Wireless Mouse,Electronics,SUP-001
PRD-002,Office Chair,Furniture,SUP-002
PRD-003,Desk Lamp,Furniture,SUP-002
```

**`orders.csv`** — note the `customer_id` and `product_id` columns; those drive `PLACED` and `CONTAINS`
```
order_id,order_date,customer_id,product_id
ORD-001,2026-06-01,CUST-001,PRD-001
ORD-002,2026-06-03,CUST-002,PRD-002
ORD-003,2026-06-05,CUST-001,PRD-003
```

This is the same 11-node, 9-relationship graph as before (3 customers, 3 products, 2 suppliers, 3 orders) — just delivered as files instead of hand-typed Cypher, which is how you'd actually receive graph data on a real project.

### Step 3 — Upload the files and build the graph with Neo4j Data Importer

Neo4j's **Data Importer** is a no-code tool that turns CSV files into a graph by letting you map columns to node labels/properties and draw relationships between them.

1. From the **Aura Console**, open your `globalmart-graph-demo` instance, then click **Import Data** (or go to **`data-importer.neo4j.io`** directly and connect it to your instance using the same URI/username/password from Step 1).
2. **Add data source** → **Upload files** → select all 4 CSVs (`customers.csv`, `suppliers.csv`, `products.csv`, `orders.csv`) at once.
3. For each file, create a **node table**:
   - `customers.csv` → node label **Customer**, key property `customer_id`
   - `suppliers.csv` → node label **Supplier**, key property `supplier_id`
   - `products.csv` → node label **Product**, key property `product_id` (map `supplier_id` as a plain property for now — you'll turn it into a relationship next)
   - `orders.csv` → node label **Order**, key property `order_id` (map `customer_id` and `product_id` as plain properties too)
4. Draw the 3 relationships by dragging between node tables in the canvas:
   - **Customer → Order**, type `PLACED`, matched on `orders.customer_id = customers.customer_id`
   - **Order → Product**, type `CONTAINS`, matched on `orders.product_id = products.product_id`
   - **Product → Supplier**, type `SUPPLIED_BY`, matched on `products.supplier_id = suppliers.supplier_id`
5. Click **Run import**. The tool writes every node and relationship into your AuraDB instance in one batch — no Cypher typed by hand.
6. Switch back to the **Neo4j Browser** (Step 1) and run `MATCH (n) RETURN n` to see the graph — confirm **11 nodes, 9 relationships**.

> If you'd rather see the raw Cypher a file-based import like this compiles down to, here's the equivalent for just the `Customer` node table and the `PLACED` relationship — the Data Importer generates and runs statements like this for you, across all 4 files, in one click:
> ```cypher
> LOAD CSV WITH HEADERS FROM 'file:///customers.csv' AS row
> CREATE (:Customer {customer_id: row.customer_id, name: row.name, city: row.city})
>
> LOAD CSV WITH HEADERS FROM 'file:///orders.csv' AS row
> MATCH (c:Customer {customer_id: row.customer_id})
> MERGE (o:Order {order_id: row.order_id, order_date: row.order_date})
> CREATE (c)-[:PLACED]->(o)
> ```

### Step 4 — Try Cypher queries in the Neo4j Browser

Paste each of these one at a time and look at the result table (or switch to the graph view):

```cypher
// Which products did each customer buy?
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN c.name AS customer_name, p.name AS product_name, o.order_id AS order_id
```

### YOUR TURN (in the Neo4j Browser)
Write and run a Cypher query that counts how many orders each customer placed, sorted highest first. (Same shape as ILT 1's pattern #4 — `MATCH ... RETURN ... COUNT(o) ... ORDER BY`.)

---
## Part 3 — Read the Graph Into Databricks

Back in this notebook now. We connect to AuraDB from Databricks using the official `neo4j` Python driver, run the same customer→product query as Step 3, and land the result as a sandbox Delta table — exactly the same shape of task as the API section, just a different source.

In [ ]:
# ─── Install the Neo4j Python driver on this cluster ───────────────────────────
%pip install neo4j

In [ ]:
# ─── Connect to your AuraDB instance ───────────────────────────────────────────
# Paste the 3 values Neo4j showed you when the instance was created in Step 1.
# NOTE: never commit real credentials — replace these placeholders locally,
# then swap them back to placeholders before you upload/submit this notebook.

from neo4j import GraphDatabase

NEO4J_URI      = "neo4j+s://YOUR_INSTANCE_ID.databases.neo4j.io"  # ← replace
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "YOUR_AURADB_PASSWORD"  # ← replace

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, params=None):
    """Run one Cypher query against AuraDB and return a list of plain dicts."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return [dict(record) for record in result]

# Quick connectivity check — should print 11 (3 customers + 3 products + 2 suppliers + 3 orders)
node_count = run_cypher("MATCH (n) RETURN count(n) AS total")[0]["total"]
print(f"Connected! Total nodes in the graph: {node_count}")

In [ ]:
# ─── Run the same customer -> product query from Step 3, but from Databricks ───

customer_product_query = """
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN c.customer_id AS customer_id, c.name AS customer_name,
       p.product_id  AS product_id,  p.name AS product_name,
       o.order_id    AS order_id,    o.order_date AS order_date
"""

rows = run_cypher(customer_product_query)
print(f"Rows returned from Neo4j: {len(rows)}")
for r in rows:
    print(r)

In [ ]:
# ─── Convert to a Spark DataFrame and save to sandbox ──────────────────────────
# Same pattern as the API section: land in sandbox/, never bronze/ — this is
# exploration output, not a GlobalMart pipeline table.

graph_df = spark.createDataFrame(rows)
graph_df.show(truncate=False)

graph_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{sandbox_path}/graph_customer_orders")

print(f"Saved to: {sandbox_path}/graph_customer_orders")

### YOUR TURN — Supplier Traversal

Write a Cypher query (as a Python string, same shape as `customer_product_query` above) that returns, for every product, which supplier it comes from — a 2-hop traversal (`Product -[:SUPPLIED_BY]-> Supplier`) plus which orders contained that product (`Order -[:CONTAINS]-> Product`). Run it through `run_cypher(...)`, convert to a DataFrame, and save it to `sandbox_path/graph_product_suppliers`.

In [ ]:
# ─── YOUR TURN: complete this query and save the result ───────────────────────

product_supplier_query = """
# TODO: MATCH a path across Order -[:CONTAINS]-> Product -[:SUPPLIED_BY]-> Supplier
# RETURN order_id, product_name, supplier_name
"""

# Uncomment once the query above is complete:
# supplier_rows = run_cypher(product_supplier_query)
# supplier_df = spark.createDataFrame(supplier_rows)
# supplier_df.show(truncate=False)
# supplier_df.write.format("delta").mode("overwrite").save(f"{sandbox_path}/graph_product_suppliers")

In [ ]:
# ─── Close the Neo4j driver connection when done ───────────────────────────────
# Good hygiene — an open driver holds a connection pool open on AuraDB's side.

driver.close()
print("Neo4j driver closed.")

---
## Submission Checklist

```
Submission Checklist
─────────────────────────────────────────────────────────
✅ Multi-base-currency FX table saved  → sandbox/fx_rates_multi_base
✅ Historical FX table saved (append mode) → sandbox/fx_rates_historical
── Rows in historical table:          ______
✅ Neo4j AuraDB Free account + instance created, credentials saved
✅ 4 CSV files prepared (customers, suppliers, products, orders)
✅ Files uploaded and imported via Neo4j Data Importer (11 nodes, 9 relationships)
✅ Customer -> Order -> Product query run in Neo4j Browser
✅ Same query run from Databricks via the neo4j Python driver
✅ Result saved → sandbox/graph_customer_orders
✅ Supplier traversal exercise completed → sandbox/graph_product_suppliers
✅ Before submitting: replaced NEO4J_URI/PASSWORD with placeholders again
─────────────────────────────────────────────────────────
```

**Reminder:** none of this feeds `fact_sales`. You practiced two ingestion *patterns* — REST API and graph traversal — that show up in other projects, using GlobalMart as a familiar backdrop.

---
## Next — Day 3 ILT 2 (2:00 PM – 3:00 PM)
**Autoloader & Schema Evolution Concepts**